In this notebook, I'll analyze the behavior from epilepsy patients. 

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import glob as glob
import arviz as az
import pandas as pd
import pickle 
import sys 
import pymc as pm
import os
from os.path import join

from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple
import arviz as az
import bambi as bmb
import pymc as pm 
from scipy.special import expit

from scipy.stats import ttest_ind

# # Hierarchical models
# from HierarchicalVKF import HierarchicalVKFModel
# from HierarchicalRW import HierarchicalRWModel
# from HierarchicalBayes import HierarchicalBayesModel
# from HierarchicalRW_Signed import HierarchicalRWModel_Signed
# from HierarchicalRW_Fictive1 import HierarchicalRWModel_Fictive1

# # sns.set_palette(custom_palette)

In [ ]:
from plot_config import set_plot_style
set_plot_style()

In [ ]:
import time, sys
from IPython.display import clear_output

def update_progress(progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = "Progress: [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
os.makedirs(save_dir_plots, exist_ok=True)


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# count the number of patients who are male
subj_df.Sex.value_counts()

In [ ]:
# print the mean and std of the age of the patients
print(subj_df.Age.mean(), subj_df.Age.std())



In [ ]:

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


In [ ]:
def _gelman_standardize(values: pd.Series) -> pd.Series:
    arr = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    mu = np.nanmean(arr)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=values.index)
    return pd.Series((arr - mu) / (2.0 * sd), index=values.index)
    
def load_behavior_tables(load_dir: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load behavior tables used by MV analyses.

    Returns
    -------
    (learn_df, combined_df)
    """
    learn_path = os.path.join(load_dir, "learn_df_RWH.csv")
    combined_path = os.path.join(load_dir, "full_df_RWH.csv")
    learn_df = pd.read_csv(learn_path)
    combined_df = pd.read_csv(combined_path)
    return learn_df, combined_df

def preprocess_learn_df(learn_df: pd.DataFrame) -> pd.DataFrame:
    """Replicates Subject-class learn_df preprocessing from mv_regression_utils.py."""
    df = learn_df.copy()

    if "gamble_rt" in df.columns and "gamble_rt_log" not in df.columns:
        with np.errstate(invalid="ignore"):
            df["gamble_rt_log"] = np.log(df["gamble_rt"])

    if {"rpe", "reward"}.issubset(df.columns):
        df["Q"] = df["rpe"] + df["reward"]

    if {"participant", "rpe"}.issubset(df.columns):
        df["prior_rpe"] = df.groupby("participant")["rpe"].shift(1)

    reversal_trials = [16, 33, 48, 65]
    if {"trials", "block"}.issubset(df.columns):
        def _tsr(row: pd.Series) -> float:
            blk = row.get("block", np.nan)
            tr = row.get("trials", np.nan)
            if not np.isfinite(blk) or not np.isfinite(tr):
                return np.nan
            if blk > 1:
                idx = int(blk - 2)
                if 0 <= idx < len(reversal_trials):
                    return float(np.mod(tr, reversal_trials[idx]))
            return float(tr)

        df["trials_since_reversal"] = df.apply(_tsr, axis=1)

    if "reward" in df.columns:
        df["preward"] = df["reward"].map({1: 0.5, 0: -0.5})

    if "gamble" in df.columns:
        df["pgamble"] = df["gamble"].map({1: 0.5, 0: -0.5})

    if "participant" in df.columns:
        for subj in df["participant"].dropna().unique():
            mask = df["participant"] == subj
            for src, dst in [
                ("Q", "zQ"),
                ("prior_rpe", "zprior_rpe"),
                ("rpe", "zrpe"),
                ("DPRIME", "zpm"),
                ("trials_since_reversal", "ztrials_since_reversal"),
            ]:
                if src in df.columns:
                    df.loc[mask, dst] = _gelman_standardize(df.loc[mask, src]).values

            if "gamble_rt_log" in df.columns and "reward" in df.columns:
                rt = pd.to_numeric(df.loc[mask, "gamble_rt_log"], errors="coerce").to_numpy(dtype=float)
                rt_min = np.nanmin(rt)
                rt_max = np.nanmax(rt)
                if np.isfinite(rt_min) and np.isfinite(rt_max) and (rt_max - rt_min) > 0:
                    rt_ev = 1.0 - ((rt - rt_min) / (rt_max - rt_min))
                    rt_rpe = pd.to_numeric(df.loc[mask, "reward"], errors="coerce").to_numpy(dtype=float) - rt_ev
                    df.loc[mask, "RT_rpe"] = rt_rpe
                    s = np.nanstd(rt_rpe)
                    if np.isfinite(s) and s > 0:
                        df.loc[mask, "zRT_rpe"] = (rt_rpe - np.nanmean(rt_rpe)) / (2.0 * s)

            if "rpe" in df.columns:
                vals = pd.to_numeric(df.loc[mask, "rpe"], errors="coerce")
                try:
                    df.loc[mask, "rpe_cat"] = pd.qcut(
                        vals,
                        q=3,
                        labels=["negative", "neutral", "positive"],
                        duplicates="drop",
                    )
                except Exception:
                    # Degenerate distributions can fail qcut.
                    df.loc[mask, "rpe_cat"] = np.nan

    return df


def preprocess_combined_df(combined_df: pd.DataFrame) -> pd.DataFrame:
    """Replicates Subject-class combined_df preprocessing from mv_regression_utils.py."""
    df = combined_df.copy()

    outcome_cols = ["hits", "misses", "correct_rejections", "false_alarms"]
    present_outcomes = [c for c in outcome_cols if c in df.columns]
    if len(present_outcomes) > 0:
        df["memory_outcome"] = df[present_outcomes].idxmax(axis=1)

    if "gamble_rt_log" not in df.columns and "gamble_rt_gamble" in df.columns:
        with np.errstate(invalid="ignore"):
            df["gamble_rt_log"] = np.log(df["gamble_rt_gamble"])

    if {"rpe", "reward_gamble"}.issubset(df.columns):
        df["Q"] = df["rpe"] + df["reward_gamble"]

    if {"participant", "rpe"}.issubset(df.columns):
        df["prior_rpe"] = df.groupby("participant")["rpe"].shift(1)

    reversal_trials = [16, 33, 48, 65]
    if {"trials_gamble", "block_gamble"}.issubset(df.columns):
        def _tsr(row: pd.Series) -> float:
            blk = row.get("block_gamble", np.nan)
            tr = row.get("trials_gamble", np.nan)
            if not np.isfinite(blk) or not np.isfinite(tr):
                return np.nan
            if blk > 1:
                idx = int(blk - 2)
                if 0 <= idx < len(reversal_trials):
                    return float(np.mod(tr, reversal_trials[idx]))
            return float(tr)

        df["trials_since_reversal"] = df.apply(_tsr, axis=1)

    if "hits" in df.columns:
        hits_bool = df["hits"].astype(bool)
        df["phit"] = np.where(hits_bool, 0.5, -0.5)

    if "reward_gamble" in df.columns:
        df["preward"] = df["reward_gamble"].map({1: 0.5, 0: -0.5})

    if "gamble_gamble" in df.columns:
        df["pgamble"] = df["gamble_gamble"].map({1: 0.5, 0: -0.5})

    if "participant" in df.columns:
        for subj in df["participant"].dropna().unique():
            mask = df["participant"] == subj
            for src, dst in [
                ("Q", "zQ"),
                ("prior_rpe", "zprior_rpe"),
                ("rpe", "zrpe"),
                ("DPRIME", "zpm"),
                ("trials_since_reversal", "ztrials_since_reversal"),
            ]:
                if src in df.columns:
                    df.loc[mask, dst] = _gelman_standardize(df.loc[mask, src]).values

            if "gamble_rt_log" in df.columns and "reward_gamble" in df.columns:
                rt = pd.to_numeric(df.loc[mask, "gamble_rt_log"], errors="coerce").to_numpy(dtype=float)
                rt_min = np.nanmin(rt)
                rt_max = np.nanmax(rt)
                if np.isfinite(rt_min) and np.isfinite(rt_max) and (rt_max - rt_min) > 0:
                    rt_ev = 1.0 - ((rt - rt_min) / (rt_max - rt_min))
                    rt_rpe = pd.to_numeric(df.loc[mask, "reward_gamble"], errors="coerce").to_numpy(dtype=float) - rt_ev
                    df.loc[mask, "RT_rpe"] = rt_rpe
                    s = np.nanstd(rt_rpe)
                    if np.isfinite(s) and s > 0:
                        df.loc[mask, "zRT_rpe"] = (rt_rpe - np.nanmean(rt_rpe)) / (2.0 * s)

            if "rpe" in df.columns:
                vals = pd.to_numeric(df.loc[mask, "rpe"], errors="coerce")
                try:
                    df.loc[mask, "rpe_cat"] = pd.qcut(
                        vals,
                        q=3,
                        labels=["neg", "neutral", "pos"],
                        duplicates="drop",
                    )
                except Exception:
                    df.loc[mask, "rpe_cat"] = np.nan

    return df

In [ ]:
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

## Figure 2A

In [ ]:

f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
sns.lineplot(data=learn_df, x='trials', y= 'male', color='orange', ax=ax, ci=68)
sns.lineplot(data=learn_df, x='trials', y= 'female', color='purple', ax=ax, ci=68)
plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
plt.ylabel('p(draw)')
plt.title(f'subject performance')
# pdf.savefig()



## Figure 2B

In [ ]:
learn_df_prolific = pd.read_csv(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_HierarchicalRW.csv')


In [ ]:
# make desnity plot comparing summed reward over participants for learn_df and learn_df_prolific
sns.kdeplot(learn_df_prolific[['participant', 'reward']].groupby('participant').mean().reward, label='Prolific', color='gray')
sns.kdeplot(learn_df[['participant', 'reward']].groupby('participant').mean().reward, label='EMU', color='black')
plt.legend()
sns.despine()
plt.savefig(f'{save_dir_plots}/Prolific_EMU_Reward_Density.pdf', dpi=300)
plt.show()



In [ ]:
# add a t-test to compare the means of the two distributions
t_test = ttest_ind(learn_df_prolific[['participant', 'reward']].groupby('participant').mean().reward, learn_df[['participant', 'reward']].groupby('participant').mean().reward)
print(t_test)


## Figure 2C

In [ ]:
ax = sns.jointplot(data = learn_df_raw.groupby('participant').mean(numeric_only=True), 
                    x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                    fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)



In [ ]:
def run_model(df, y=None, X=None, Intx=None, rand_effect=None, rand_slopes=False, within_subject_X=None,
              within_subject_Intx=None, priors=None, categorical=None,
              cores=2, chains=2, tune=1500, draws=2000, target_accept=0.95, model_fam='bernoulli', 
              output_dir='/Volumes/T7 Shield/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest', return_model=False,
              categorical_baseline=True, save_model_res = True, label=None):
    """
    Run the model: 
    
    Parameters
    ----------
    df: pandas dataframe 
        All the variables 
    y: str
        Name of the dependent variable
    X: list of strings 
        Names of the independent variables 
    Intx: list of strings
        Names of the interactions, formatted as 'a:b' 
    rand_effect: str
        Name of variable for random effects (i.e. 'subject') 
    rand_slopes: bool 
        Should we fit random slopes in addition to random intercepts? 
    priors: dict 
        Prior distribution for any independent variables (i.e. Intercept, random intercept) 
        Default to weakly informative priors (Gelman et al. 2008) 
    categorical: list of strings 
        Names of categorical variables going into model 
    
    Returns
    -------
    results: model
    
    
    y = 'memory_outcome'
    X = ['sds_score', 'zrpe', 'zpm', 'gamble_rt_log', 'recog_rt_log', 'img_cat', 'age', 'Sex', 'alpha_pos', 'alpha_neg', 'beta']
    Intx = ['sds_score:zrpe', 'sds_score:zpm']
    rand_effect = ['participant']
    categorical = ['img_cat', 'Sex']

    # Drop nan data
    model_df = model_df.dropna(subset=X)

    model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=False, 
                               priors=None, categorical=categorical, cores=4, chains=4, tune=1500, draws=2000, 
                               target_accept=0.9, model_fam='bernoulli', 
                               output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                               return_model=True,
                               categorical_baseline=True, 
                               save_model_res = True, 
                               label=None)
    
    """

    
    # if no label provided, make your own
    if not label: 
        label = (f"{y}" + "_{}"*len(X)).format(*X)
        
    
    if categorical_baseline: # this means we set a baseline for categorical effects 
        # format random effects properly 
        rand_term = [f'(1|{x})' for x in rand_effect]
        if rand_slopes: # do we want a more complex mixed-effects model with random slopes as well as intercepts?
            if not within_subject_X:
                raise ValueError('No within subject variable specified')
            if not Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])
            elif not within_subject_Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'({intx}|{rand_effect[0]})' for intx in within_subject_Intx])
        else:
            if not Intx: 
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)
    else: # this means all levels of categorical effect are judged absent or present 
         # format random effects properly 
        rand_term = [f'(0+1|{x})' for x in rand_effect]
        if rand_slopes: 
            formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'(0 + {x}|{rand_effect[0]})' for x in X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'(0 + {intx}|{rand_effect[0]})' for intx in Intx])
        else:
            if not Intx:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)        

    print(formula)
    # construct the model 
    model = bmb.Model(formula=formula, 
                  data=df[rand_effect + [y] + X],
                 family=model_fam,
                 priors=priors,
                 categorical=categorical)
    
    # fit the model 
    results=model.fit(cores=cores, 
                          chains=chains, 
                          tune=tune, 
                          draws=draws, 
                          target_accept=target_accept)
    # init='advi+adapt_diag'

    # save out the CSV with the results 
    summary = az.summary(results, hdi_prob=0.95)
    for var in summary.index.values:
        if var.endswith("]"):  # skip individual subject parameters
            continue
        if var.startswith("1"):  # skip random effects.
            continue
        summary.loc[var, "P>0"] = np.mean(results.posterior[var].values > 0)
    summary.to_csv(join(output_dir , f"{label}_summary.csv"))

    # save out the HDI plot of all the fixed effects 
    axes = az.plot_forest(results,
                           kind='ridgeplot',
                           var_names=[f'^{x}' for x in X],
                           filter_vars="regex",
                           colors='black',
                           combined=True,
                           hdi_prob=0.95,
                           figsize=(9, 7))
    plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
    # forestplot
#         plt.xlim([-0.2, 0.1])

    plt.savefig(join(output_dir, f'{label}_HDIplot.pdf'), dpi=300)
    plt.close()

    # print out the latex table for the cell output
    results_df = summary.loc[:,
                   ['mean', 'sd', 'hdi_2.5%', 'hdi_97.5%']
                  ]
    all_tests = [] 

    for x in X: 
        temp = results_df.filter(like=f'{x}', axis=0)
        all_tests.append(temp)

    results_df = pd.concat(all_tests)

    c_string = '|c'*results_df.shape[0] + '|'
#     print(results_df.reset_index().to_latex(index=False, 
#                                             column_format=c_string).replace("\\\n", "\\ \hline\n"))  

    if save_model_res:
        # save out the model for model comparison (THIS WILL BE GB-sized, so really only needed for model comparison)
        az.to_netcdf(results, join(output_dir , f"{label}_model"))
    
    if return_model:
        return model, results 

        
def plot_res(fitted_model, X, ax=None, save_label=None): 
    """
    Plot selected posterior effects sorted from most positive mean to least.
    Effects with 95% HDI excluding 0 are colored blue.
    """

    import re
    import scipy.stats as st

    summary = az.summary(fitted_model, hdi_prob=0.95)

    rows = []
    for x in X:
        pattern = re.compile(rf'^(?!.*\|).*{re.escape(x)}.*$')
        matching = [v for v in summary.index if pattern.search(v)]
        for var in matching:
            rows.append({
                'var': var,
                'mean': float(summary.loc[var, 'mean']),
                'hdi_lo': float(summary.loc[var, 'hdi_2.5%']),
                'hdi_hi': float(summary.loc[var, 'hdi_97.5%']),
            })

    if len(rows) == 0:
        raise ValueError('No matching posterior variables found for X.')

    res_df = pd.DataFrame(rows).drop_duplicates('var')
    res_df['significant'] = (res_df['hdi_lo'] > 0) | (res_df['hdi_hi'] < 0)
    res_df = res_df.sort_values('mean', ascending=False).reset_index(drop=True)

    if ax is None:
        _, ax = plt.subplots(figsize=(9, 0.45 * len(res_df) + 2))

    y_positions = np.arange(len(res_df))

    for i, row in res_df.iterrows():
        vals = fitted_model.posterior[row['var']].values.ravel()
        kde = st.gaussian_kde(vals)
        xs = np.linspace(vals.min(), vals.max(), 400)
        dens = kde(xs)
        dens = dens / dens.max() * 0.35

        color = 'tab:blue' if row['significant'] else 'gray'
        y = y_positions[i]

        # One-sided ridge (non-symmetric density)
        ax.fill_between(xs, y, y + dens, color=color, alpha=0.45, linewidth=0)
        ax.hlines(y, row['hdi_lo'], row['hdi_hi'], color=color, linewidth=2.5)
        ax.plot(row['mean'], y, 'o', color=color, markersize=4)

    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(res_df['var'])
    ax.set_ylim(-0.25, len(res_df) - 0.25 + 0.45)
    ax.set_xlabel('Posterior effect size')
    ax.set_ylabel('Predictor')
    sns.despine(ax=ax, left=True)

    if save_label:
        plt.savefig(save_label, dpi=300, bbox_inches='tight')

    plt.show()
    return res_df

# def plot_res(fitted_model, X, ax=None, save_label=None): 
#     """
#     Code for plotting select beta weights in a publication friendly manner from saved csv
#     """
        
#     axes = az.plot_forest(fitted_model,
#                            kind='ridgeplot',
#                            var_names=[f'^(?!.*\|).*{x}.*$' for x in X],
#                            filter_vars="regex",
#                            colors='gray',
#                            combined=True,
#                            ridgeplot_overlap=2,
#                            ridgeplot_truncate=False,
#                            ridgeplot_alpha=0.4,
#                            hdi_prob=0.95,
#                            figsize=(9, 7),
#                            ax=ax)
#     # 'forestplot'
#     plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
#     plt.show()
# #     plt.savefig(save_label, dpi=300)
# #     plt.close()

In [ ]:
subj_df

## Figure 2D

In [ ]:
model_df = combined_df[
    (combined_df.condition == 'Day1')
].dropna(subset=['hits', 'rpe', 'participant']).copy()

print(model_df.shape)
subj_df.rename(columns={'MSSMCode':'participant',
'Age':'age', 'Sex':'sex'}, inplace=True)
model_df.drop(columns=['age', 'sex'], inplace=True)
model_df = model_df.merge(subj_df, on='participant')
model_df.rename(columns={'DPRIME_x':'pm'}, inplace=True)
model_df = model_df[['participant', 'hits', 'rpe', 'pm', 'sex', 'age', 'alpha', 'beta']]

In [ ]:
# Prep a list to store results 
all_res = [] 
# Define the dependent variable and the independent variables
dependent_var = 'hits'
standardize_within_subject = ['rpe', 'pm']
standardize_between_subject = ['alpha', 'beta', 'age']
binary_standardize = ['sex']

for factor in standardize_within_subject: 
  for subj in model_df.participant.unique(): 
    subj_data = model_df[model_df.participant==subj]
    subj_data[f'z{factor}'] = _gelman_standardize(subj_data[factor])
    model_df.loc[model_df.participant==subj, f'z{factor}'] = subj_data[f'z{factor}']

for factor in standardize_between_subject: 
  model_df[f'z{factor}'] = _gelman_standardize(model_df[factor])

for factor in binary_standardize: 
  model_df[f'z{factor}'] = model_df[factor].apply(lambda x: 0.5 if x == 'M' else -0.5)

model_df = model_df.dropna()


In [ ]:
y = 'hits'
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
within_subject_X = ['zrpe']
Intx = []
rand_effect = ['participant']
categorical = None

label = (f"{y}" + "_{}"*len(X)).format(*X) + 'emu_manuscript'

model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/Volumes/T7 Shield/scratch/MemoryBandit/bambi_models', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
plot_res(results, X, ax=ax, save_label=f'{save_dir_plots}/EMUPosteriors_Manuscript.pdf')


## Figure 2E

MATLAB

## Figure 2F

Schematic